# 🧩 Mini-Lab: Fail Loud, Retry Only Transients

**Module 7: Trust but Verify** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Replace** a bare `except` with narrow `AIClientError` / `ValidationError` handling
2. **Retry** transient errors only, with a cap, backoff, and per-run budget
3. **Fail loud** on permanent errors instead of swallowing them

## Target Concepts

| Concept | Gloss |
|---------|-------|
| Narrow exception handling | Custom exception types beat bare `except`; never swallow errors |
| Bounded retry | Retry transient errors only; cap attempts, wall-clock, and budget |


## The Problem: Silent Failures

A **bare `except`** catches *every* exception — including bugs you need to see. The program keeps running, but with wrong or missing data. You only discover the problem hours later (or never).

| Pattern | Risk |
|---------|------|
| `except:` | Catches `KeyboardInterrupt`, `SystemExit`, *everything* |
| `except Exception:` | Catches all bugs — `TypeError`, `NameError`, `IndexError` — and hides them |
| `except Exception: pass` | The worst: swallows the error *and* produces no output |

## Step 1 — See a Silent Failure in Action

Below is a function that parses a price from a string. It has a **bug** (a typo in the variable name), but the bare `except` hides it completely.

In [1]:
def parse_price_bad(text: str) -> float:
    """Parse a dollar price from text — BAD version with bare except."""
    try:
        cleaned = text.replace("$", "").replace(",", "").strip()
        price = float(cleaned)
        return pirce  # ← typo! should be 'price'
    except:
        return 0.0  # silently returns 0 for ALL errors


# This LOOKS like it works — but every result is wrong
prices = ["$19.99", "$1,250.00", "free", "$42.50"]
results = [parse_price_bad(p) for p in prices]
print("Parsed prices:", results)
print("Total:        ", sum(results))

Parsed prices: [0.0, 0.0, 0.0, 0.0]
Total:         0.0


Every price came back as `0.0` — even the valid ones. The `NameError` from the typo (`pirce`) was silently swallowed. In a real pipeline, you'd compute a total of `$0.00` and never know why.

## Step 2 — Define a Narrow Custom Exception

Instead of catching everything, we define a custom exception that describes the *specific* failure we can anticipate: a price string that can't be converted to a number.

In [2]:
class PriceParsingError(ValueError):
    """Raised when a price string cannot be parsed into a number."""

    def __init__(self, raw_value: str, reason: str):
        self.raw_value = raw_value
        self.reason = reason
        super().__init__(f"Cannot parse price '{raw_value}': {reason}")


print("Custom exception defined:", PriceParsingError.__mro__)

Custom exception defined: (<class '__main__.PriceParsingError'>, <class 'ValueError'>, <class 'Exception'>, <class 'BaseException'>, <class 'object'>)


Notice two design choices:

1. **Inherits from `ValueError`**, not `Exception` — this tells readers *what kind* of problem it is
2. **Carries context** (`raw_value`, `reason`) — the error message tells you exactly what input failed and why

## Step 3 — Catch Narrow, Let Bugs Propagate

Now we rewrite `parse_price` to catch **only** `ValueError` (the one error we expect from `float()`) and raise our custom exception. Any other error — like the typo — will propagate and be visible immediately.

In [3]:
def parse_price(text: str) -> float:
    """Parse a dollar price from text — GOOD version with narrow except."""
    cleaned = text.replace("$", "").replace(",", "").strip()
    if not cleaned:
        raise PriceParsingError(text, "empty after cleaning")
    try:
        return float(cleaned)
    except ValueError:
        raise PriceParsingError(text, f"'{cleaned}' is not a valid number")

Key differences from the bad version:

- The `try` block wraps **only** the `float()` call — the smallest possible scope
- We catch **only** `ValueError` — the one exception `float()` raises on bad input
- A typo, `TypeError`, or any other bug is **not caught** — it crashes loudly so you fix it

## Step 4 — Test with Good and Bad Inputs

Let's run the same price list through the improved function. This time, valid prices succeed and invalid ones produce a clear, specific error.

In [4]:
prices = ["$19.99", "$1,250.00", "free", "$42.50", ""]

for p in prices:
    try:
        result = parse_price(p)
        print(f"  ✅ '{p}' → {result}")
    except PriceParsingError as exc:
        print(f"  ❌ {exc}")

  ✅ '$19.99' → 19.99
  ✅ '$1,250.00' → 1250.0
  ❌ Cannot parse price 'free': 'free' is not a valid number
  ✅ '$42.50' → 42.5
  ❌ Cannot parse price '': empty after cleaning


Valid prices are parsed correctly; invalid ones produce a `PriceParsingError` with the exact input and reason. No silent zeros.

## Step 5 — The Caller Decides What to Do

With narrow exceptions, the **caller** can choose the right recovery strategy: skip, use a default, or abort. The function doesn't make that decision for you.

In [5]:
prices = ["$19.99", "free", "$42.50", "N/A", "$8.00"]

# Strategy: skip bad values, collect only valid ones
valid_prices = []
skipped = []

for p in prices:
    try:
        valid_prices.append(parse_price(p))
    except PriceParsingError:
        skipped.append(p)

print(f"Valid prices: {valid_prices}")
print(f"Skipped:      {skipped}")
print(f"Total:        ${sum(valid_prices):.2f}")

Valid prices: [19.99, 42.5, 8.0]
Skipped:      ['free', 'N/A']
Total:        $70.49


## Step 6 — Side-by-Side Comparison

Let's summarize the two approaches.

In [6]:
from IPython.display import display, Markdown

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

md("""
| | ❌ Bare `except` | ✅ Narrow `except` |
|---|---|---|
| **What it catches** | Everything — bugs, typos, system signals | Only the specific error you expect |
| **Bugs** | Hidden — program continues with wrong data | Visible — crash immediately so you fix them |
| **Error message** | None (or generic) | Specific: what input failed and why |
| **Caller control** | None — function decides to return a default | Full — caller picks the recovery strategy |
| **Debugging** | "Why is the total $0?" (hours later) | `PriceParsingError: 'free' is not a valid number` (immediately) |
""")


| | ❌ Bare `except` | ✅ Narrow `except` |
|---|---|---|
| **What it catches** | Everything — bugs, typos, system signals | Only the specific error you expect |
| **Bugs** | Hidden — program continues with wrong data | Visible — crash immediately so you fix them |
| **Error message** | None (or generic) | Specific: what input failed and why |
| **Caller control** | None — function decides to return a default | Full — caller picks the recovery strategy |
| **Debugging** | "Why is the total $0?" (hours later) | `PriceParsingError: 'free' is not a valid number` (immediately) |


---
## Retry only transients

Fail loud on permanent errors. Cap retries, time, and budget.


## Why Bounded Retries?

Network calls fail. APIs return 500s, connections time out, rate limits kick in. The instinct to "just retry" is correct — but **how** you retry matters enormously:

| Pattern | Risk |
|---------|------|
| **No retry** | One transient glitch = total failure |
| **Retry forever** | Hung process, runaway costs, server gets hammered |
| **Retry N times, no backoff** | Floods the server with rapid requests |
| ✅ **Bounded retry with backoff + budget** | Recovers from transient errors, gives up safely |

## Step 1 — Define Retryable vs. Permanent Errors

Recall from `mini-narrow-exceptions`: we catch only what we can recover from. For retries, **transient** errors (timeouts, server errors) deserve another attempt. **Permanent** errors (bad request, auth failure) should fail immediately — retrying won't help.

In [1]:
class TransientError(Exception):
    """A temporary failure that may succeed on retry (timeout, 500, 429)."""
    def __init__(self, message: str, status_code: int | None = None):
        self.status_code = status_code
        super().__init__(message)


class PermanentError(Exception):
    """A failure that will NOT succeed on retry (400, 401, 403)."""
    def __init__(self, message: str, status_code: int | None = None):
        self.status_code = status_code
        super().__init__(message)


class BudgetExhaustedError(Exception):
    """Raised when the total time budget for retries has been exceeded."""


print("Exception types defined:")
print(f"  TransientError  → retry")
print(f"  PermanentError  → fail immediately")
print(f"  BudgetExhaustedError → no more time left")

Exception types defined:
  TransientError  → retry
  PermanentError  → fail immediately
  BudgetExhaustedError → no more time left


## Step 2 — Build a Fake API for Testing

We need a controllable "API" that fails in predictable ways so we can test the retry logic without making real network calls. This simulator takes a script of status codes and returns them in order.

In [2]:
import time


class FakeAPI:
    """Simulates an HTTP API that returns pre-scripted status codes."""

    def __init__(self, responses: list[int]):
        self.responses = list(responses)
        self.call_count = 0

    def call(self) -> dict:
        """Simulate an API call. Raises on non-200 status codes."""
        self.call_count += 1
        if not self.responses:
            raise PermanentError("No more scripted responses", status_code=500)

        status = self.responses.pop(0)

        if status == 200:
            return {"status": 200, "data": "success"}
        elif status in (429, 500, 502, 503):
            raise TransientError(f"Server returned {status}", status_code=status)
        elif status == 408:
            raise TransientError("Request timed out", status_code=408)
        else:
            raise PermanentError(f"Client error {status}", status_code=status)


# Quick test: 500, then 200
api = FakeAPI([500, 200])
try:
    api.call()
except TransientError as e:
    print(f"Call 1: {e}  (retryable)")

result = api.call()
print(f"Call 2: {result}  (success)")

Call 1: Server returned 500  (retryable)
Call 2: {'status': 200, 'data': 'success'}  (success)


## Step 3 — Implement Bounded Retry

Our retry function enforces **four safety bounds**:

1. **Max attempts** — stop after N tries
2. **Exponential backoff** — wait longer between each attempt (1s, 2s, 4s, ...)
3. **Jitter** — add randomness so multiple callers don't all retry at the same instant
4. **Time budget** — a hard wall-clock limit for the entire operation (all attempts combined)

In [3]:
import random


def retry_with_budget(
    fn,
    max_attempts: int = 4,
    base_delay: float = 1.0,
    budget_seconds: float = 30.0,
):
    """
    Call fn() with bounded retries on TransientError.

    - max_attempts:   most times to try (including the first call)
    - base_delay:     initial backoff in seconds (doubles each retry)
    - budget_seconds: total wall-clock time allowed for all attempts
    """
    start = time.monotonic()

    for attempt in range(1, max_attempts + 1):
        elapsed = time.monotonic() - start

        # --- Budget check BEFORE each attempt ---
        if elapsed >= budget_seconds:
            raise BudgetExhaustedError(
                f"Time budget of {budget_seconds}s exhausted after {attempt - 1} attempts"
            )

        try:
            return fn()  # success → return immediately

        except PermanentError:
            raise  # not retryable → propagate immediately

        except TransientError as exc:
            if attempt == max_attempts:
                raise  # last attempt → give up

            # Exponential backoff + jitter
            delay = base_delay * (2 ** (attempt - 1))
            jitter = random.uniform(0, delay * 0.5)
            wait = delay + jitter

            # Don't wait longer than remaining budget
            remaining = budget_seconds - (time.monotonic() - start)
            if wait > remaining:
                wait = max(remaining, 0)

            print(
                f"  Attempt {attempt}/{max_attempts} failed: {exc} "
                f"→ retrying in {wait:.2f}s"
            )
            time.sleep(wait)


print("retry_with_budget() defined")

retry_with_budget() defined


Let's walk through the key design choices:

| Feature | How it works |
|---------|-------------|
| **Narrow catch** | Only `TransientError` triggers a retry; `PermanentError` is re-raised immediately |
| **Exponential backoff** | Delay doubles each attempt: 1s → 2s → 4s → ... |
| **Jitter** | Random 0–50% of the delay is added, preventing thundering-herd retries |
| **Budget cap** | Total elapsed time is checked before each attempt; wait time is clamped to remaining budget |
| **Max attempts** | Hard stop after N tries even if budget remains |

## Step 4 — Scenario: Transient Failure then Success

The most common case — the server hiccups once or twice but recovers.

In [4]:
# Script: 500, 503, then 200 on third attempt
api = FakeAPI([500, 503, 200])

# Use small delays for demo (real code would use 1-2s base)
result = retry_with_budget(api.call, max_attempts=4, base_delay=0.1, budget_seconds=10.0)

print(f"\n✅ Success after {api.call_count} attempts: {result}")

  Attempt 1/4 failed: Server returned 500 → retrying in 0.11s


  Attempt 2/4 failed: Server returned 503 → retrying in 0.23s



✅ Success after 3 attempts: {'status': 200, 'data': 'success'}


## Step 5 — Scenario: Permanent Error Stops Immediately

A 401 (unauthorized) is permanent — retrying with the same bad credentials won't help. The retry loop should **not** waste attempts.

In [5]:
api = FakeAPI([401])

try:
    retry_with_budget(api.call, max_attempts=4, base_delay=0.1, budget_seconds=10.0)
except PermanentError as exc:
    print(f"❌ Stopped immediately (no retries): {exc}")
    print(f"   Total API calls made: {api.call_count}")

❌ Stopped immediately (no retries): Client error 401
   Total API calls made: 1


## Step 6 — Scenario: Max Attempts Exhausted

If the server keeps failing, we give up after `max_attempts` rather than retrying forever.

In [6]:
# Server returns 500 every time — never recovers
api = FakeAPI([500, 500, 500, 500, 500])

try:
    retry_with_budget(api.call, max_attempts=3, base_delay=0.1, budget_seconds=30.0)
except TransientError as exc:
    print(f"\n❌ Gave up after {api.call_count} attempts: {exc}")

  Attempt 1/3 failed: Server returned 500 → retrying in 0.11s
  Attempt 2/3 failed: Server returned 500 → retrying in 0.25s



❌ Gave up after 3 attempts: Server returned 500


## Step 7 — Scenario: Budget Exhausted

Even if `max_attempts` allows more tries, the **time budget** is the hard ceiling. This prevents a slow-responding server from keeping your process hung.

In [7]:
# 500s with a very tight budget — backoff will eat up the budget before max_attempts
api = FakeAPI([500, 500, 500, 500, 500, 500, 500])

try:
    retry_with_budget(
        api.call,
        max_attempts=7,       # plenty of attempts allowed
        base_delay=0.5,       # backoff grows: 0.5s, 1s, 2s, 4s ...
        budget_seconds=2.0,   # but only 2 seconds total!
    )
except (TransientError, BudgetExhaustedError) as exc:
    print(f"\n❌ {type(exc).__name__}: {exc}")
    print(f"   Total API calls made: {api.call_count}")

  Attempt 1/7 failed: Server returned 500 → retrying in 0.71s


  Attempt 2/7 failed: Server returned 500 → retrying in 1.12s


  Attempt 3/7 failed: Server returned 500 → retrying in 0.17s

❌ BudgetExhaustedError: Time budget of 2.0s exhausted after 3 attempts
   Total API calls made: 3


## Step 8 — All Four Bounds at a Glance

Let's summarize what stops the retry loop in each scenario.

In [8]:
from IPython.display import display, Markdown

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

md("""
| Bound | What stops the loop | When it matters |
|-------|--------------------|-----------------|
| **Max attempts** | Hard cap on number of tries | Prevents infinite retries on a persistently broken server |
| **Exponential backoff** | Increasing wait between tries | Gives the server time to recover; reduces load |
| **Jitter** | Random addition to delay | Prevents multiple clients from retrying in sync (thundering herd) |
| **Time budget** | Total wall-clock cap for all attempts | Prevents a hung process when backoff delays accumulate |

And the **narrow catch** ensures:
- `TransientError` → retry (the server might recover)
- `PermanentError` → fail immediately (retrying won't help)
- Any other exception → propagates (it's a bug, not a server issue)
""")


| Bound | What stops the loop | When it matters |
|-------|--------------------|-----------------|
| **Max attempts** | Hard cap on number of tries | Prevents infinite retries on a persistently broken server |
| **Exponential backoff** | Increasing wait between tries | Gives the server time to recover; reduces load |
| **Jitter** | Random addition to delay | Prevents multiple clients from retrying in sync (thundering herd) |
| **Time budget** | Total wall-clock cap for all attempts | Prevents a hung process when backoff delays accumulate |

And the **narrow catch** ensures:
- `TransientError` → retry (the server might recover)
- `PermanentError` → fail immediately (retrying won't help)
- Any other exception → propagates (it's a bug, not a server issue)


## 🎯 Summary

### Key Takeaways

1. **Silent failure** — a bare `except` swallows every error including bugs, letting the program continue with wrong data and no warning
2. **Narrow exception handling** — catch only the specific exception type you can recover from (e.g., `ValueError` from `float()`), and let everything else propagate
3. **Custom exception types** — defining a class like `PriceParsingError(ValueError)` carries context (what input, why it failed) and makes errors self-documenting
4. **Caller decides recovery** — the function raises; the caller chooses whether to skip, default, retry, or abort